In [1]:
import pandas as pd
import os

DATA_ROOT = "/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/filtered_dataset/data_partitions/"
ROOT_VAL_REF = DATA_ROOT + "valid_frac20/"

separamos al azar las 15, digamos 12+3, entonces después replicamos las 12 hasta llegar a 80 y tenemos en train, y replicamos las 3 hasta llegar a 20 y tenemos el valid

In [2]:
ls = os.listdir(ROOT_VAL_REF)
ls

['ArchiveII_fam-fold.csv',
 'ArchiveII_hc_400.csv',
 'ArchiveII_rnadist_100.csv',
 'ArchiveII_hc_100.csv',
 'ArchiveII_samples_400.csv',
 'ArchiveII_samples_200.csv',
 'ArchiveII_hc_200.csv',
 'ArchiveII_rnadist_400.csv',
 'ArchiveII_samples_100.csv',
 'ArchiveII_rnadist_200.csv']

In [4]:
data = ls[3]
df_ref = pd.read_csv(ROOT_VAL_REF + data)
print(df_ref)

       fold partition                            id
0       16s      test        16s_A.fulgidus_domain2
1       16s      test        16s_A.fulgidus_domain3
2       16s      test        16s_A.fulgidus_domain4
3       16s      test      16s_A.pyrophilus_domain2
4       16s      test      16s_A.pyrophilus_domain4
...     ...       ...                           ...
9379  tmRNA     valid    telomerase_AF221916.94-481
9380  tmRNA     valid    telomerase_AF221930.94-546
9381  tmRNA     valid    telomerase_AF221931.99-541
9382  tmRNA     valid    telomerase_AF221936.98-540
9383  tmRNA     valid  telomerase_AY312571.605-1069

[9384 rows x 3 columns]


In [17]:
df_ref = pd.read_csv(ROOT_VAL_REF + data)
df_ref["fam"] = df_ref["id"].str.partition("_")[0]
df_ref

,fold,partition,id,fam
0,16s,test,16s_A.fulgidus_domain2,16s
1,16s,test,16s_A.fulgidus_domain3,16s
2,16s,test,16s_A.fulgidus_domain4,16s
3,16s,test,16s_A.pyrophilus_domain2,16s
4,16s,test,16s_A.pyrophilus_domain4,16s
...,...,...,...,...
9379,tmRNA,valid,telomerase_AF221916.94-481,telomerase
9380,tmRNA,valid,telomerase_AF221930.94-546,telomerase
9381,tmRNA,valid,telomerase_AF221931.99-541,telomerase
9382,tmRNA,valid,telomerase_AF221936.98-540,telomerase


In [18]:
df_ref.groupby(["fold", "partition"])["id"].nunique()

fold        partition
16s         test           66
            train         499
            valid         125
23s         test           15
            train         540
            valid         135
5s          test         1283
            train         472
            valid         118
RNaseP      test          454
            train         472
            valid         118
grp1        test           74
            train         492
            valid         124
srp         test          918
            train         472
            valid         118
tRNA        test          557
            train         472
            valid         118
telomerase  test           35
            train         524
            valid         131
tmRNA       test          462
            train         472
            valid         118
Name: id, dtype: int64

In [19]:
df_ref.query('partition == "test"').groupby(["fold"])["id"].count()

fold
16s             66
23s             15
5s            1283
RNaseP         454
grp1            74
srp            918
tRNA           557
telomerase      35
tmRNA          462
Name: id, dtype: int64

In [20]:
tr = (
    df_ref.query('partition == "train" & fold == "5s"')
    .groupby(["fold", "fam"])["id"]
    .count()
)
val = (
    df_ref.query('partition == "valid" & fold == "5s"')
    .groupby(["fold", "fam"])["id"]
    .count()
)

In [21]:
tr

fold  fam       
5s    16s           53
      23s           12
      RNaseP        80
      grp1          59
      srp           80
      tRNA          80
      telomerase    28
      tmRNA         80
Name: id, dtype: int64

In [22]:
val

fold  fam       
5s    16s           13
      23s            3
      RNaseP        20
      grp1          15
      srp           20
      tRNA          20
      telomerase     7
      tmRNA         20
Name: id, dtype: int64

In [23]:
tr + val

fold  fam       
5s    16s            66
      23s            15
      RNaseP        100
      grp1           74
      srp           100
      tRNA          100
      telomerase     35
      tmRNA         100
Name: id, dtype: int64

In [24]:
df_ref.query('partition == "test"').groupby(["fold"])["id"].nunique()

fold
16s             66
23s             15
5s            1283
RNaseP         454
grp1            74
srp            918
tRNA           557
telomerase      35
tmRNA          462
Name: id, dtype: int64

In [42]:
import pandas as pd


def data_partition_oversampling_80_20(df, target=100, train_split=0.8):
    """
    df: pd.DataFrame [fold, id, partition]
    """
    # Extraer familia
    df["fam"] = df["id"].str.partition("_")[0]
    fams = df["fam"].unique()
    # para cada fold de entrenamiento (fold == familia en test)
    df_collector = []
    for fold in fams:
        df_train = df[(df["partition"] == "train") & (df["fold"] == fold)]
        df_valid = df[(df["partition"] == "valid") & (df["fold"] == fold)]

        df_test = df[(df["partition"] == "test") & (df["fold"] == fold)]
        # para cada familia posible reviso que todas tengan 100 elementos
        for f in fams:
            if f != fold:
                df_t_fam = df_train[df_train["fam"] == f]
                df_v_fam = df_valid[df_valid["fam"] == f]

                df_t_fam = data_cycle_oversampling(
                    df_t_fam, int(round(target * train_split))
                )
                df_v_fam = data_cycle_oversampling(
                    df_v_fam, int(round(target * (1 - train_split)))
                )
                df_collector.append(df_t_fam)
                df_collector.append(df_v_fam)
            # tambien colecto test para unirlo todo
            else:
                df_collector.append(df_test)
        df_final = pd.concat(df_collector, ignore_index=True)
    print(
        f"old shape: {df.shape[0]}",
        f"new shape: {df_final.shape[0]}",
        f"new_shape/old_shape {round(df_final.shape[0] / df.shape[0], 2)}",
    )
    return df_final[["fold", "partition", "id"]]  # quito fam


def data_cycle_oversampling(df: "pd.DataFrame", target: int = 100) -> "pd.DataFrame":
    """Return a copy of df padded with its own rows (in order, cyclically)
    until it reaches target rows.
    """
    df_out = df.copy()
    if df_out.shape[0] >= target:
        return df_out

    i = 0
    while df_out.shape[0] < target:
        df_out = pd.concat([df_out, df.iloc[[i]]], ignore_index=True)
        i = (i + 1) % len(df)
    return df_out


# Aca hay que hacer que este df de test reemplace la familia en train.

In [43]:
SAVE = False

SAVE_PATH = DATA_ROOT + "over_sampled_w_valid20/"
methods = ["rnadist", "hc", "samples"]
methods = ["rnadist"]
thresholds = [100, 200, 400]

for method in methods:
    for thershold in thresholds:
        file = f"ArchiveII_{method}_{thershold}.csv"
        print(file[:-4])
        df = pd.read_csv(ROOT_VAL_REF + file)
        df = data_partition_oversampling_80_20(df, thershold)
        if SAVE:
            df.to_csv(SAVE_PATH + file)

ArchiveII_rnadist_100
old shape: 9384 new shape: 11064 new_shape/old_shape 1.18
ArchiveII_rnadist_200
old shape: 13384 new shape: 18264 new_shape/old_shape 1.36
ArchiveII_rnadist_400
old shape: 21384 new shape: 32664 new_shape/old_shape 1.53


In [58]:
df.duplicated().sum()

np.int64(11280)

In [56]:
df.groupby(["partition", "fold"])["id"].count().sort_values()

partition  fold      
test       23s             15
           telomerase      35
           16s             66
           grp1            74
           RNaseP         454
           tmRNA          462
           tRNA           557
valid      tmRNA          640
           tRNA           640
           telomerase     640
           srp            640
           grp1           640
           5s             640
           RNaseP         640
           16s            640
           23s            640
test       srp            918
           5s            1283
train      grp1          2560
           srp           2560
           5s            2560
           23s           2560
           16s           2560
           RNaseP        2560
           tRNA          2560
           telomerase    2560
           tmRNA         2560
Name: id, dtype: int64

## Check sanity

In [34]:
df["fam"] = df["id"].str.partition("_")[0]
tr = (
    df.query('partition == "train" & fold == "5s"')
    .groupby(["fold", "fam"])["id"]
    .count()
)

val = (
    df.query('partition == "valid" & fold == "5s"')
    .groupby(["fold", "fam"])["id"]
    .count()
)

In [35]:
tr

fold  fam       
5s    16s           80
      23s           80
      RNaseP        80
      grp1          80
      srp           80
      tRNA          80
      telomerase    80
      tmRNA         80
Name: id, dtype: int64

In [36]:
val

fold  fam       
5s    16s           20
      23s           20
      RNaseP        20
      grp1          20
      srp           20
      tRNA          20
      telomerase    20
      tmRNA         20
Name: id, dtype: int64

In [37]:
tr + val

fold  fam       
5s    16s           100
      23s           100
      RNaseP        100
      grp1          100
      srp           100
      tRNA          100
      telomerase    100
      tmRNA         100
Name: id, dtype: int64

In [31]:
df_ref.groupby(["fold", "partition"])["id"].nunique()

fold        partition
16s         test           66
            train         499
            valid         125
23s         test           15
            train         540
            valid         135
5s          test         1283
            train         472
            valid         118
RNaseP      test          454
            train         472
            valid         118
grp1        test           74
            train         492
            valid         124
srp         test          918
            train         472
            valid         118
tRNA        test          557
            train         472
            valid         118
telomerase  test           35
            train         524
            valid         131
tmRNA       test          462
            train         472
            valid         118
Name: id, dtype: int64

In [32]:
df.groupby(["fold", "partition"])["id"].count()

fold        partition
16s         test           66
            train        2560
            valid         640
23s         test           15
            train        2560
            valid         640
5s          test         1283
            train        2560
            valid         640
RNaseP      test          454
            train        2560
            valid         640
grp1        test           74
            train        2560
            valid         640
srp         test          918
            train        2560
            valid         640
tRNA        test          557
            train        2560
            valid         640
telomerase  test           35
            train        2560
            valid         640
tmRNA       test          462
            train        2560
            valid         640
Name: id, dtype: int64